### Menu Knowledge Agent

Creates an Agent Bricks Knowledge Assistant over the restaurant menu PDFs
stored in the Unity Catalog volume. The Knowledge Assistant uses Databricks'
Instructed Retriever to answer questions with citations.

In [ ]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
ENDPOINT_NAME = dbutils.widgets.get("MENU_KNOWLEDGE_ENDPOINT_NAME")

In [ ]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

VOLUME_PATH = f"/Volumes/{CATALOG}/menu_documents/menus"
AGENT_NAME = f"{CATALOG}-menu-knowledge"
API_BASE = "/api/2.0/knowledge-assistants"

KA_BODY = {
    "name": AGENT_NAME,
    "description": (
        "Answers questions about Caspers Kitchens restaurant menus "
        "including dishes, ingredients, nutrition, allergens, and preparation details. "
        "Covers all 16 restaurant brands."
    ),
    "endpoint_name": ENDPOINT_NAME,
    "knowledge_sources": [
        {
            "files_source": {
                "name": "menu_pdfs",
                "type": "files",
                "files": {"path": VOLUME_PATH},
                "description": (
                    "Restaurant menu PDFs for 16 ghost kitchen brands. "
                    "Each PDF contains the full menu with item names, descriptions, "
                    "prices, nutritional information (calories, protein, fat, carbs), "
                    "and allergen warnings."
                ),
            },
        },
    ],
    "instructions": (
        "You are a restaurant menu assistant for Caspers Kitchens. "
        "Always cite which brand the information comes from. "
        "Be specific about allergen warnings and nutritional details."
    ),
}

def get_tile_by_name(name):
    """Look up a tile by name from /api/2.0/tiles. Returns the tile dict or None."""
    try:
        resp = w.api_client.do("GET", "/api/2.0/tiles")
        for tile in resp.get("tiles", []):
            if tile.get("name") == name:
                return tile
    except Exception:
        pass
    return None

def ka_exists(ka_id):
    """Return True only if the KA ID is live in the API."""
    try:
        w.api_client.do("GET", f"{API_BASE}/{ka_id}")
        return True
    except Exception:
        return False

def find_existing_id():
    """Look up agent ID from uc_state."""
    try:
        df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type IN ('knowledge_assistants', 'endpoints')
            ORDER BY created_at DESC
        """)
        for row in df.collect():
            info = json.loads(row.resource_data)
            if info.get("endpoint_name") == ENDPOINT_NAME:
                return info.get("tile_id") or info.get("agent_id")
    except Exception:
        pass
    return None

agent_id = None
needs_polling = True

# Path 1: found ID in uc_state — verify it's still live before updating
existing_id = find_existing_id()
if existing_id:
    if ka_exists(existing_id):
        try:
            w.api_client.do("PUT", f"{API_BASE}/{existing_id}", body=KA_BODY)
            agent_id = existing_id
            needs_polling = False
            print(f"✅ Updated existing Menu KA: {agent_id}")
        except Exception as e:
            print(f"  PUT failed for {existing_id}: {e} — will create new")
            agent_id = None
    else:
        print(f"  Stale uc_state ID {existing_id} (not found in API) — will create new")

# Path 2: tile exists by name — verify it's still live before updating
if not agent_id:
    tile = get_tile_by_name(AGENT_NAME)
    if tile and ka_exists(tile["tile_id"]):
        try:
            agent_id = tile["tile_id"]
            w.api_client.do("PUT", f"{API_BASE}/{agent_id}", body=KA_BODY)
            needs_polling = False
            print(f"♻️ Updated existing Menu KA by name: {agent_id}")
        except Exception as e:
            print(f"  PUT by name failed: {e} — will create new")
            agent_id = None
    elif tile:
        print(f"  Stale tile {tile['tile_id']} (not found in API) — will create new")

# Path 3: create new
if not agent_id:
    try:
        w.api_client.do("POST", API_BASE, body=KA_BODY)
        print(f"✅ Created Menu Knowledge Assistant")
    except Exception as e:
        err = str(e)
        if "already exists" in err.lower() or "ALREADY_EXISTS" in err:
            print(f"♻️ Agent {AGENT_NAME} already exists (name collision), fetching ID")
        else:
            raise

    # Resolve tile_id — POST response doesn't include it
    import time as _time
    tile = get_tile_by_name(AGENT_NAME)
    if not tile:
        _time.sleep(10)
        tile = get_tile_by_name(AGENT_NAME)
    if tile:
        agent_id = tile["tile_id"]
        print(f"  Resolved tile_id: {agent_id}")
    else:
        agent_id = AGENT_NAME
        needs_polling = False
        print(f"⚠️ Could not resolve tile_id, using name as fallback")

print(f"   Agent ID (tile_id): {agent_id}")
print(f"   Endpoint: {ENDPOINT_NAME}")

In [ ]:
import time

if needs_polling and agent_id and agent_id != AGENT_NAME:
    MAX_WAIT = 600
    POLL_INTERVAL = 30
    elapsed = 0
    print(f"Polling Menu KA readiness (id={agent_id}, max {MAX_WAIT}s)...")

    while elapsed < MAX_WAIT:
        try:
            ka_status = w.api_client.do("GET", f"{API_BASE}/{agent_id}")
            ep_status = ka_status.get("knowledge_assistant", {}).get("status", {}).get("endpoint_status", "")
            print(f"  [{elapsed}s] endpoint_status: {ep_status}")
            if str(ep_status).upper() in ("ONLINE", "ACTIVE", "READY"):
                print(f"✅ Menu Knowledge Assistant is READY")
                break
        except Exception as e:
            print(f"  [{elapsed}s] poll error: {type(e).__name__}: {e}")

        time.sleep(POLL_INTERVAL)
        elapsed += POLL_INTERVAL
    else:
        print(f"⏳ Endpoint may still be provisioning — proceeding.")
else:
    print(f"✅ Menu Knowledge Assistant already running — skipping polling.")

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

add(CATALOG, "knowledge_assistants", {"endpoint_name": ENDPOINT_NAME, "tile_id": agent_id, "name": AGENT_NAME})
print("\u2705 Menu Knowledge Agent stage complete")